# Divvy Bike-Share Data Lakehouse on Azure Databricks

**Flow:** Unity Catalog Volume (source CSV) → Bronze Delta files (volume) → registered Bronze Delta tables → Gold Delta star schema → analytics

**Business questions**
1. Ride time by weekday / time of day, start & end station, rider age at trip, member vs casual.
2. Payments per month / quarter / year, and per member by age at account start.
3. *(Extra credit)* Spend per member vs. average rides per month and monthly riding minutes.

**Star schema** (see `Divvy_Star_Schema.pdf`): `fact_trip`, `fact_payment`; dimensions `dim_rider`, `dim_station` (role-played as start/end), `dim_date`.

**Before running:** upload `riders.csv`, `payments.csv`, `stations.csv`, `trips.csv` to `<volume>/raw/` in Catalog Explorer, attach a Unity Catalog compute (single node, DBR 13.3 LTS+), edit the three names in the next cell, then *Run all*.

**Source note:** the four CSV files have **no header row**, so column names come from the ERD and are applied explicitly below.

## 0. Configuration

In [0]:
from pyspark.sql import functions as F, Window

# ---- EDIT THESE THREE to match Catalog Explorer (catalog > schema > volume) --------------
CATALOG       = "databricks_306026"      
VOLUME_SCHEMA = "default"   # schema that holds the volume
VOLUME        = "divvy"     # volume name
# -------------------------------------------------------------------------------------------
BRONZE_SCHEMA = "divvy_bronze"   # registered Bronze tables
GOLD_SCHEMA   = "divvy_gold"     # Gold star schema tables

VOLUME_BASE = f"/Volumes/{CATALOG}/{VOLUME_SCHEMA}/{VOLUME}"
RAW_DIR     = f"{VOLUME_BASE}/raw"            # source CSVs (never written to by this notebook)
BRONZE_DIR  = f"{VOLUME_BASE}/bronze_delta"   # Bronze Delta *files* (one folder per entity)

SOURCE_FILES = {"rider": "riders.csv", "payment": "payments.csv",
                "station": "stations.csv", "trip": "trips.csv"}

# Column names + target types. The CSVs have no header, so this is the single source of truth.
BRONZE_COLS = {
    "rider":   [("rider_id", "INT"), ("first", "STRING"), ("last", "STRING"), ("address", "STRING"),
                ("birthday", "DATE"), ("account_start_date", "DATE"), ("account_end_date", "DATE"),
                ("is_member", "BOOLEAN")],
    "payment": [("payment_id", "INT"), ("date", "DATE"), ("amount", "DECIMAL(10,2)"), ("rider_id", "INT")],
    "station": [("station_id", "STRING"), ("name", "STRING"), ("latitude", "DOUBLE"), ("longitude", "DOUBLE")],
    "trip":    [("trip_id", "STRING"), ("rideable_type", "STRING"), ("started_at", "TIMESTAMP"),
                ("ended_at", "TIMESTAMP"), ("start_station_id", "STRING"), ("end_station_id", "STRING"),
                ("rider_id", "INT")],
}

def tbl(schema, name):
    """Fully qualified table name catalog.schema.table."""
    return f"{CATALOG}.{schema}.{name}" if CATALOG else f"{schema}.{name}"

def ls(path):
    return [f.name for f in dbutils.fs.ls(path)]

try:
    display
except NameError:                      # lets the same code run outside Databricks (local test)
    display = lambda df: df.show(20, False)

print("Volume base :", VOLUME_BASE)
print("Spark       :", spark.version)

Volume base : /Volumes/databricks_306026/default/divvy
Spark       : 3.5.0


In [0]:
# Validate that the four source files exist in the volume before doing anything else.
found = ls(RAW_DIR)
missing = [f for f in SOURCE_FILES.values() if f not in found]
assert not missing, f"Missing in {RAW_DIR}: {missing}. Found: {found}. Check CATALOG / VOLUME_SCHEMA / VOLUME and the upload."
print("OK - all source files found:", sorted(SOURCE_FILES.values()))

OK - all source files found: ['payments.csv', 'riders.csv', 'stations.csv', 'trips.csv']


**Source → Bronze → Gold mapping**

| Source file | Bronze table | Columns (from ERD; files have no header) | Gold use |
|---|---|---|---|
| `riders.csv` | `rider` | rider_id, first, last, address, birthday, account_start_date, account_end_date, is_member | `dim_rider` (names/address are dropped: not needed) |
| `stations.csv` | `station` | station_id *(string: values like `KA1503000012`)*, name, latitude, longitude | `dim_station` |
| `trips.csv` | `trip` | trip_id, rideable_type, started_at, ended_at, start_station_id, end_station_id, rider_id | `fact_trip` |
| `payments.csv` | `payment` | payment_id, date, amount, rider_id | `fact_payment` |

ERD correction: `station_id` / `start_station_id` / `end_station_id` are **strings**, not INT as drawn in the ERD. There is no separate Account file; account fields live on the rider.

## 1. Source inspection
Read every CSV as **strings** with an explicit column list (no schema inference), then convert with `try_cast`, which yields NULL instead of failing. Any value that was present in the file but became NULL is counted as a **parse failure**.

In [0]:
def read_raw(name):
    ddl = ", ".join(f"`{c}` STRING" for c, _ in BRONZE_COLS[name])
    return (spark.read.option("header", "false").option("mode", "PERMISSIVE")
            .schema(ddl).csv(f"{RAW_DIR}/{SOURCE_FILES[name]}"))

def type_and_check(name):
    """Return (typed DataFrame, {column: parse_failure_count})."""
    cols = BRONZE_COLS[name]
    typed = [c if t == "STRING" else f"try_cast(`{c}` AS {t}) AS `{c}`" for c, t in cols]
    nonstr = [c for c, t in cols if t != "STRING"]
    both = read_raw(name).selectExpr(*typed, *[f"`{c}` AS `_raw_{c}`" for c in nonstr])
    fails = both.agg(*[F.sum(F.when(F.col(f"_raw_{c}").isNotNull() & F.col(c).isNull(), 1).otherwise(0)).alias(c)
                       for c in nonstr]).first().asDict()
    return both.select(*[c for c, _ in cols]), {k: int(v or 0) for k, v in fails.items()}

In [0]:
KEYS = {"rider": "rider_id", "payment": "payment_id", "station": "station_id", "trip": "trip_id"}
inspection = {}
for name in BRONZE_COLS:
    df, fails = type_and_check(name)
    cols = [c for c, _ in BRONZE_COLS[name]]
    stats = df.agg(F.count(F.lit(1)).alias("rows"),
                   F.countDistinct(KEYS[name]).alias("distinct_keys"),
                   *[F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}") for c in cols]).first().asDict()
    inspection[name] = (stats, fails)
    print(f"\n=== {name}  ({SOURCE_FILES[name]}) ===")
    df.printSchema()
    display(df.limit(5))
    print("rows:", stats["rows"], "| distinct keys:", stats["distinct_keys"],
          "| duplicate keys:", stats["rows"] - stats["distinct_keys"])
    print("nulls per column:", {c: stats[f"null_{c}"] for c in cols if stats[f"null_{c}"]})
    print("parse failures  :", {c: n for c, n in fails.items() if n} or "none")

bad = {n: {c: v for c, v in f.items() if v} for n, (_, f) in inspection.items() if any(f.values())}
assert not bad, f"Type conversion failures (header row present? wrong format?): {bad}"


=== rider  (riders.csv) ===
root
 |-- rider_id: integer (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- address: string (nullable = true)
 |-- birthday: date (nullable = true)
 |-- account_start_date: date (nullable = true)
 |-- account_end_date: date (nullable = true)
 |-- is_member: boolean (nullable = true)



rider_id,first,last,address,birthday,account_start_date,account_end_date,is_member
1000,[redacted],[redacted],[redacted],1989-02-13,2019-04-23,null,true
1001,[redacted],[redacted],[redacted],1976-08-10,2019-11-01,2020-09-01,true
1002,[redacted],[redacted],[redacted],1998-08-10,2022-02-04,null,true
1003,[redacted],[redacted],[redacted],1999-03-29,2019-08-26,null,false
1004,[redacted],[redacted],[redacted],1969-04-11,2019-09-14,null,true


rows: 75000 | distinct keys: 75000 | duplicate keys: 0
nulls per column: {'account_end_date': 60046}
parse failures  : none

=== payment  (payments.csv) ===
root
 |-- payment_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- rider_id: integer (nullable = true)



payment_id,date,amount,rider_id
1,2019-05-01,9.00,1000
2,2019-06-01,9.00,1000
3,2019-07-01,9.00,1000
4,2019-08-01,9.00,1000
5,2019-09-01,9.00,1000


rows: 1946607 | distinct keys: 1946607 | duplicate keys: 0
nulls per column: {}
parse failures  : none

=== station  (stations.csv) ===
root
 |-- station_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



station_id,name,latitude,longitude
525,Glenwood Ave & Touhy Ave,42.012701,-87.66605799999999
KA1503000012,Clark St & Lake St,41.88579466666667,-87.63110066666668
637,Wood St & Chicago Ave,41.895634,-87.672069
13216,State St & 33rd St,41.8347335,-87.6258275
18003,Fairbanks St & Superior St,41.89580766666667,-87.62025316666669


rows: 838 | distinct keys: 838 | duplicate keys: 0
nulls per column: {}
parse failures  : none

=== trip  (trips.csv) ===
root
 |-- trip_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- rider_id: integer (nullable = true)



trip_id,rideable_type,started_at,ended_at,start_station_id,end_station_id,rider_id
89E7AA6C29227EFF,classic_bike,2021-02-12T16:14:56Z,2021-02-12T16:21:43Z,525,660,71934
0FEFDE2603568365,classic_bike,2021-02-14T17:52:38Z,2021-02-14T18:12:09Z,525,16806,47854
E6159D746B2DBB91,electric_bike,2021-02-09T19:10:18Z,2021-02-09T19:19:10Z,KA1503000012,TA1305000029,70870
B32D3199F1C2E75B,classic_bike,2021-02-02T17:49:41Z,2021-02-02T17:54:06Z,637,TA1305000034,58974
83E463F23575F4BF,electric_bike,2021-02-23T15:07:23Z,2021-02-23T15:22:37Z,13216,TA1309000055,39608


rows: 4584921 | distinct keys: 4584921 | duplicate keys: 0
nulls per column: {}
parse failures  : none


*What to look for:* `rider.account_end_date` is legitimately NULL for active accounts; every other column should have 0 nulls. Parse failures must be `none`. If a file turns out to have a header row, the failure counts will point at the ID column.

## 2. Extract: write Bronze Delta **files** to the volume
Each entity is written to its own folder under `bronze_delta/`. The `raw/` folder with the CSVs is never touched. `mode("overwrite")` makes re-runs idempotent.

In [0]:
CSV_COUNTS = {n: inspection[n][0]["rows"] for n in BRONZE_COLS}
for name in BRONZE_COLS:
    df, _ = type_and_check(name)
    (df.write.format("delta").mode("overwrite").save(f"{BRONZE_DIR}/{name}"))
    print("wrote", f"{BRONZE_DIR}/{name}")

wrote /Volumes/databricks_306026/default/divvy/bronze_delta/rider
wrote /Volumes/databricks_306026/default/divvy/bronze_delta/payment
wrote /Volumes/databricks_306026/default/divvy/bronze_delta/station
wrote /Volumes/databricks_306026/default/divvy/bronze_delta/trip


In [0]:
# Verify: Delta transaction log exists, then read back and compare counts with the CSV.
FILE_COUNTS = {}
for name in BRONZE_COLS:
    path = f"{BRONZE_DIR}/{name}"
    assert "_delta_log/" in ls(path), f"No _delta_log in {path}"
    FILE_COUNTS[name] = spark.read.format("delta").load(path).count()
    print(f"{name:8s} csv={CSV_COUNTS[name]:>10,}  delta_files={FILE_COUNTS[name]:>10,}  match={CSV_COUNTS[name] == FILE_COUNTS[name]}")
assert CSV_COUNTS == FILE_COUNTS
print(ls(f"{BRONZE_DIR}/trip")[:5])

rider    csv=    75,000  delta_files=    75,000  match=True
payment  csv= 1,946,607  delta_files= 1,946,607  match=True
station  csv=       838  delta_files=       838  match=True
trip     csv= 4,584,921  delta_files= 4,584,921  match=True
['_delta_log/', 'part-00000-e997d03b-ea6f-477e-8c14-61a80d89dca0-c000.snappy.parquet', 'part-00001-87798245-8572-4d5f-9f29-dedf66b08a4b-c000.snappy.parquet', 'part-00002-ded98228-ff61-4eb4-a8f1-c74efdd9aa31-c000.snappy.parquet', 'part-00003-bb639c97-e940-45af-bf5d-f675df2989e1-c000.snappy.parquet']


## 3. Load: registered Bronze Delta tables (Spark SQL)
Unity Catalog does **not** allow a table `LOCATION` inside a volume (path overlap), so the Bronze *tables* are managed Delta tables in Unity Catalog storage. They are created with `CREATE TABLE` and filled with `INSERT OVERWRITE` from the Delta files written in section 2 (via temporary views).

In [0]:
for schema in (BRONZE_SCHEMA, GOLD_SCHEMA):
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}" if CATALOG else f"CREATE SCHEMA IF NOT EXISTS {schema}")
    except Exception as e:   # no CREATE SCHEMA privilege: use an existing schema instead
        raise RuntimeError(f"Cannot create schema {schema}. Pick an existing writable schema and set "
                           f"BRONZE_SCHEMA / GOLD_SCHEMA above. Original error: {e}")
print("schemas ready:", BRONZE_SCHEMA, GOLD_SCHEMA)

schemas ready: divvy_bronze divvy_gold


In [0]:
for name, cols in BRONZE_COLS.items():
    t = tbl(BRONZE_SCHEMA, name)
    col_ddl = ", ".join(f"`{c}` {ty}" for c, ty in cols)
    col_list = ", ".join(f"`{c}`" for c, _ in cols)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {t} ({col_ddl}) USING DELTA")
    spark.read.format("delta").load(f"{BRONZE_DIR}/{name}").createOrReplaceTempView(f"v_bronze_{name}")
    spark.sql(f"INSERT OVERWRITE {t} SELECT {col_list} FROM v_bronze_{name}")
    print("loaded", t)

loaded databricks_306026.divvy_bronze.rider
loaded databricks_306026.divvy_bronze.payment
loaded databricks_306026.divvy_bronze.station
loaded databricks_306026.divvy_bronze.trip


In [0]:
TABLE_COUNTS = {}
for name in BRONZE_COLS:
    t = tbl(BRONZE_SCHEMA, name)
    detail = spark.sql(f"DESCRIBE DETAIL {t}").select("format", "numFiles", "sizeInBytes").first()
    TABLE_COUNTS[name] = spark.sql(f"SELECT COUNT(*) AS n FROM {t}").first()["n"]
    print(f"{t:40s} format={detail['format']}  files={detail['numFiles']}  rows={TABLE_COUNTS[name]:,}")
    assert detail["format"] == "delta" and TABLE_COUNTS[name] == FILE_COUNTS[name]
display(spark.sql(f"SELECT * FROM {tbl(BRONZE_SCHEMA, 'trip')} LIMIT 5"))

databricks_306026.divvy_bronze.rider     format=delta  files=1  rows=75,000
databricks_306026.divvy_bronze.payment   format=delta  files=1  rows=1,946,607
databricks_306026.divvy_bronze.station   format=delta  files=1  rows=838
databricks_306026.divvy_bronze.trip      format=delta  files=4  rows=4,584,921


trip_id,rideable_type,started_at,ended_at,start_station_id,end_station_id,rider_id
89E7AA6C29227EFF,classic_bike,2021-02-12T16:14:56Z,2021-02-12T16:21:43Z,525,660,71934
0FEFDE2603568365,classic_bike,2021-02-14T17:52:38Z,2021-02-14T18:12:09Z,525,16806,47854
E6159D746B2DBB91,electric_bike,2021-02-09T19:10:18Z,2021-02-09T19:19:10Z,KA1503000012,TA1305000029,70870
B32D3199F1C2E75B,classic_bike,2021-02-02T17:49:41Z,2021-02-02T17:54:06Z,637,TA1305000034,58974
83E463F23575F4BF,electric_bike,2021-02-23T15:07:23Z,2021-02-23T15:22:37Z,13216,TA1309000055,39608


## 4. Gold dimensions (PySpark)
* **Keys:** `rider_key` / `station_key` = `row_number()` over the business ID (deterministic for the same data); `date_key` = `yyyyMMdd`.
* **Duplicates:** deduplicated by business key with an explicit tie-break; keys whose duplicate rows *disagree* are counted and reported.
* **Age** = completed years (birthday anniversary respected), NULL if a date is missing or the event is before birth.
* Dimensions hold attributes only, no facts. Names/addresses are intentionally left out (PII not needed).

In [0]:
def bronze(name):
    return spark.table(tbl(BRONZE_SCHEMA, name))

def age_years(event_date, birth_date):
    """Completed years at event_date; NULL when either date is NULL or the event precedes birth."""
    e, b = F.col(event_date) if isinstance(event_date, str) else event_date, F.col(birth_date)
    before_bday = (F.month(e) * 100 + F.dayofmonth(e)) < (F.month(b) * 100 + F.dayofmonth(b))
    return (F.when(e.isNull() | b.isNull() | (e < b), F.lit(None))
             .otherwise(F.year(e) - F.year(b) - F.when(before_bday, 1).otherwise(0))).cast("int")

def dedupe(df, key, order_cols):
    """One row per key. Tie-break = order_cols (first wins). Returns (df, number of keys with conflicting rows)."""
    conflicts = df.dropDuplicates().groupBy(key).count().filter("count > 1").count()
    w = Window.partitionBy(key).orderBy(*order_cols)
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn"), conflicts

def build_dim_rider():
    r, conflicts = dedupe(bronze("rider"), "rider_id",
                          [F.col("account_start_date").desc_nulls_last(), F.col("account_end_date").desc_nulls_last()])
    dim = r.select(
        F.row_number().over(Window.orderBy("rider_id")).alias("rider_key"),
        "rider_id",
        F.col("birthday").alias("birth_date"),
        "account_start_date", "account_end_date",
        F.col("is_member"),                       # current snapshot flag, not historical status
        age_years("account_start_date", "birthday").alias("age_at_account_start"))
    return dim, conflicts

def build_dim_station():
    s, conflicts = dedupe(bronze("station"), "station_id", [F.col("name").asc_nulls_last(), F.col("latitude").asc_nulls_last()])
    dim = s.select(F.row_number().over(Window.orderBy("station_id")).alias("station_key"),
                   "station_id", F.col("name").alias("station_name"), "latitude", "longitude")
    return dim, conflicts

def build_dim_date():
    t, p = bronze("trip"), bronze("payment")
    b = (t.agg(F.min(F.to_date("started_at")).alias("t0"), F.max(F.to_date("ended_at")).alias("t1"))
          .crossJoin(p.agg(F.min("date").alias("p0"), F.max("date").alias("p1")))
          .select(F.least("t0", "p0").alias("lo"), F.greatest("t1", "p1").alias("hi")).first())
    d = (spark.range(1).select(F.explode(F.sequence(F.lit(b["lo"]), F.lit(b["hi"]), F.expr("interval 1 day"))).alias("date")))
    return d.select(
        F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
        "date",
        (F.expr("weekday(date)") + 1).alias("day_of_week_num"),          # 1 = Monday ... 7 = Sunday
        F.date_format("date", "EEEE").alias("day_of_week_name"),
        F.dayofmonth("date").alias("day_of_month"),
        F.month("date").alias("month"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.quarter("date").alias("quarter"),
        F.year("date").alias("year"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        (F.expr("weekday(date)") >= 5).alias("is_weekend"))

## 5. Gold facts (PySpark)
* `fact_trip` — grain: one row per `trip_id`. Left joins to the dimensions so no trip is dropped; unmatched station/rider keys stay NULL and are reported in validation.
* `fact_payment` — grain: one row per `payment_id`.
* Trip duration keeps fractional minutes; `duration_quality` flags `NEGATIVE` / `ZERO` rather than silently deleting rows. Timestamps are used exactly as stored in the source (no timezone assumption).
* Trips and payments are **never joined to each other** row-by-row; for the extra-credit analysis each is aggregated to rider-month first.

In [0]:
def build_fact_trip(dim_rider, dim_station):
    rider = F.broadcast(dim_rider.select("rider_key", "rider_id", "birth_date"))
    s_start = F.broadcast(dim_station.select(F.col("station_id").alias("start_station_id"), F.col("station_key").alias("start_station_key")))
    s_end = F.broadcast(dim_station.select(F.col("station_id").alias("end_station_id"), F.col("station_key").alias("end_station_key")))
    secs = F.col("ended_at").cast("long") - F.col("started_at").cast("long")
    hr = F.hour("started_at")
    return (bronze("trip")
        .join(rider, "rider_id", "left")
        .join(s_start, "start_station_id", "left")
        .join(s_end, "end_station_id", "left")
        .select(
            "trip_id", "rider_key",
            F.date_format("started_at", "yyyyMMdd").cast("int").alias("start_date_key"),
            F.date_format("ended_at", "yyyyMMdd").cast("int").alias("end_date_key"),
            "start_station_key", "end_station_key", "rideable_type", "started_at", "ended_at",
            hr.alias("start_hour"),
            F.when(hr < 6, "Night (00-05)").when(hr < 12, "Morning (06-11)")
             .when(hr < 18, "Afternoon (12-17)").otherwise("Evening (18-23)").alias("time_band"),
            secs.alias("ride_duration_seconds"),
            (secs / 60.0).alias("ride_duration_minutes"),
            F.when(secs < 0, "NEGATIVE").when(secs == 0, "ZERO").otherwise("OK").alias("duration_quality"),
            age_years(F.to_date("started_at"), "birth_date").alias("rider_age_at_trip")))

def build_fact_payment(dim_rider):
    rider = F.broadcast(dim_rider.select("rider_key", "rider_id"))
    return (bronze("payment").join(rider, "rider_id", "left")
        .select("payment_id", "rider_key",
                F.date_format("date", "yyyyMMdd").cast("int").alias("payment_date_key"),
                F.col("date").alias("payment_date"),
                F.col("amount").cast("decimal(10,2)").alias("amount")))

## 6. Gold Delta managed tables
Every Gold table is written with `write.format("delta").mode("overwrite").saveAsTable(...)`. Re-running replaces the content (no appends, nothing is dropped). Facts are built from the *saved* dimension tables so keys always match.

In [0]:
GOLD_TABLES = ["dim_rider", "dim_station", "dim_date", "fact_trip", "fact_payment"]
DIM_CONFLICTS = {}

def gold(name):
    return spark.table(tbl(GOLD_SCHEMA, name))

def write_gold(df, name):
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(tbl(GOLD_SCHEMA, name)))

def build_gold():
    dim_rider, DIM_CONFLICTS["rider"] = build_dim_rider()
    dim_station, DIM_CONFLICTS["station"] = build_dim_station()
    write_gold(dim_rider, "dim_rider")
    write_gold(dim_station, "dim_station")
    write_gold(build_dim_date(), "dim_date")
    write_gold(build_fact_trip(gold("dim_rider"), gold("dim_station")), "fact_trip")
    write_gold(build_fact_payment(gold("dim_rider")), "fact_payment")
    return {n: gold(n).count() for n in GOLD_TABLES}

GOLD_COUNTS_RUN1 = build_gold()
print(GOLD_COUNTS_RUN1)
print("keys with conflicting duplicate source rows:", DIM_CONFLICTS)

{'dim_rider': 75000, 'dim_station': 838, 'dim_date': 3288, 'fact_trip': 4584921, 'fact_payment': 1946607}
keys with conflicting duplicate source rows: {'rider': 0, 'station': 0}


In [0]:
for n in GOLD_TABLES:
    t = tbl(GOLD_SCHEMA, n)
    fmt = spark.sql(f"DESCRIBE DETAIL {t}").first()["format"]
    assert fmt == "delta" and GOLD_COUNTS_RUN1[n] > 0
    print(f"{t:38s} format={fmt} rows={GOLD_COUNTS_RUN1[n]:,}")
display(gold("fact_trip").limit(5)); display(gold("fact_payment").limit(5))
display(gold("dim_rider").limit(5)); display(gold("dim_station").limit(5)); display(gold("dim_date").limit(5))

databricks_306026.divvy_gold.dim_rider format=delta rows=75,000
databricks_306026.divvy_gold.dim_station format=delta rows=838
databricks_306026.divvy_gold.dim_date  format=delta rows=3,288
databricks_306026.divvy_gold.fact_trip format=delta rows=4,584,921
databricks_306026.divvy_gold.fact_payment format=delta rows=1,946,607


trip_id,rider_key,start_date_key,end_date_key,start_station_key,end_station_key,rideable_type,started_at,ended_at,start_hour,time_band,ride_duration_seconds,ride_duration_minutes,duration_quality,rider_age_at_trip
89E7AA6C29227EFF,70935,20210212,20210212,454,529,classic_bike,2021-02-12T16:14:56Z,2021-02-12T16:21:43Z,16,Afternoon (12-17),407,6.783333333333333,OK,37
0FEFDE2603568365,46855,20210214,20210214,454,192,classic_bike,2021-02-14T17:52:38Z,2021-02-14T18:12:09Z,17,Afternoon (12-17),1171,19.516666666666666,OK,38
E6159D746B2DBB91,69871,20210209,20210209,552,693,electric_bike,2021-02-09T19:10:18Z,2021-02-09T19:19:10Z,19,Evening (18-23),532,8.866666666666667,OK,33
B32D3199F1C2E75B,57975,20210202,20210202,512,696,classic_bike,2021-02-02T17:49:41Z,2021-02-02T17:54:06Z,17,Afternoon (12-17),265,4.416666666666667,OK,19
83E463F23575F4BF,38609,20210223,20210223,75,827,electric_bike,2021-02-23T15:07:23Z,2021-02-23T15:22:37Z,15,Afternoon (12-17),914,15.233333333333333,OK,71


payment_id,rider_key,payment_date_key,payment_date,amount
1,1,20190501,2019-05-01,9.00
2,1,20190601,2019-06-01,9.00
3,1,20190701,2019-07-01,9.00
4,1,20190801,2019-08-01,9.00
5,1,20190901,2019-09-01,9.00


rider_key,rider_id,birth_date,account_start_date,account_end_date,is_member,age_at_account_start
1,1000,1989-02-13,2019-04-23,null,true,30
2,1001,1976-08-10,2019-11-01,2020-09-01,true,43
3,1002,1998-08-10,2022-02-04,null,true,23
4,1003,1999-03-29,2019-08-26,null,false,20
5,1004,1969-04-11,2019-09-14,null,true,50


station_key,station_id,station_name,latitude,longitude
1,13001,Michigan Ave & Washington St,41.8839840647265,-87.6246839761734
2,13006,LaSalle St & Washington St,41.882664,-87.63253
3,13008,Millennium Park,41.8810317,-87.62408432
4,13011,Canal St & Adams St,41.879255,-87.639904
5,13016,St. Clair St & Erie St,41.89434513742426,-87.62279838323593


date_key,date,day_of_week_num,day_of_week_name,day_of_month,month,month_name,quarter,year,year_month,is_weekend
20130201,2013-02-01,5,Friday,1,2,February,1,2013,2013-02,false
20130202,2013-02-02,6,Saturday,2,2,February,1,2013,2013-02,true
20130203,2013-02-03,7,Sunday,3,2,February,1,2013,2013-02,true
20130204,2013-02-04,1,Monday,4,2,February,1,2013,2013-02,false
20130205,2013-02-05,2,Tuesday,5,2,February,1,2013,2013-02,false


## 7. Validation

In [0]:
results = []
def check(name, ok, detail=""):
    results.append((name, bool(ok), str(detail)))

ft, fp, dr, ds, dd = (gold(n) for n in ["fact_trip", "fact_payment", "dim_rider", "dim_station", "dim_date"])

# Row-count lineage: CSV -> Bronze files -> Bronze tables -> Gold
check("trip: csv = bronze files = bronze table = fact_trip",
      CSV_COUNTS["trip"] == FILE_COUNTS["trip"] == TABLE_COUNTS["trip"] == GOLD_COUNTS_RUN1["fact_trip"], GOLD_COUNTS_RUN1["fact_trip"])
check("payment: csv = bronze files = bronze table = fact_payment",
      CSV_COUNTS["payment"] == FILE_COUNTS["payment"] == TABLE_COUNTS["payment"] == GOLD_COUNTS_RUN1["fact_payment"], GOLD_COUNTS_RUN1["fact_payment"])
check("rider: csv = ... = dim_rider", CSV_COUNTS["rider"] == TABLE_COUNTS["rider"] == GOLD_COUNTS_RUN1["dim_rider"], GOLD_COUNTS_RUN1["dim_rider"])
check("station: csv = ... = dim_station", CSV_COUNTS["station"] == TABLE_COUNTS["station"] == GOLD_COUNTS_RUN1["dim_station"], GOLD_COUNTS_RUN1["dim_station"])

# Unique keys (also proves joins did not multiply fact rows)
for df, key, nm in [(ft, "trip_id", "fact_trip.trip_id"), (fp, "payment_id", "fact_payment.payment_id"),
                    (dr, "rider_key", "dim_rider.rider_key"), (dr, "rider_id", "dim_rider.rider_id"),
                    (ds, "station_key", "dim_station.station_key"), (ds, "station_id", "dim_station.station_id"),
                    (dd, "date_key", "dim_date.date_key")]:
    dups = df.groupBy(key).count().filter("count > 1").count()
    check(f"unique {nm}", dups == 0, f"duplicate keys: {dups}")
check("no conflicting duplicate source rows in dimensions", sum(DIM_CONFLICTS.values()) == 0, DIM_CONFLICTS)

# Unmatched / NULL references
n = lambda df, cond: df.filter(cond).count()
check("fact_trip.rider_key not NULL", n(ft, "rider_key IS NULL") == 0, n(ft, "rider_key IS NULL"))
check("fact_payment.rider_key not NULL", n(fp, "rider_key IS NULL") == 0, n(fp, "rider_key IS NULL"))
check("fact_trip station keys not NULL", n(ft, "start_station_key IS NULL OR end_station_key IS NULL") == 0,
      f"start NULL={n(ft, 'start_station_key IS NULL')}, end NULL={n(ft, 'end_station_key IS NULL')}  (any NULL = station id absent from stations.csv)")
check("fact_trip.rider_age_at_trip not NULL", n(ft, "rider_age_at_trip IS NULL") == 0, n(ft, "rider_age_at_trip IS NULL"))
check("every date key exists in dim_date",
      ft.join(dd.select(F.col("date_key").alias("k")), ft.start_date_key == F.col("k"), "left_anti").count() == 0 and
      ft.join(dd.select(F.col("date_key").alias("k")), ft.end_date_key == F.col("k"), "left_anti").count() == 0 and
      fp.join(dd.select(F.col("date_key").alias("k")), fp.payment_date_key == F.col("k"), "left_anti").count() == 0)
# Informational (a finding, not a failure): impossible durations are flagged, not removed
q = {r["duration_quality"]: r["n"] for r in ft.groupBy("duration_quality").agg(F.count("*").alias("n")).collect()}
check("duration_quality counts (info: NEGATIVE/ZERO are flagged, excluded from analytics)", True, q)
check("rider_age_at_trip in 0..110", n(ft, "rider_age_at_trip < 0 OR rider_age_at_trip > 110") == 0)

# Payment amounts: numeric and reconcile with the CSV
csv_sum = type_and_check("payment")[0].agg(F.sum("amount")).first()[0]
fact_sum = fp.agg(F.sum("amount")).first()[0]
check("payment amount: CSV sum = fact_payment sum", csv_sum == fact_sum, f"csv={csv_sum} fact={fact_sum}")
check("payment amount: no NULL / non-positive", n(fp, "amount IS NULL OR amount <= 0") == 0, n(fp, "amount IS NULL OR amount <= 0"))

# Gold overwrite is idempotent: run the whole Gold build a second time, compare counts
GOLD_COUNTS_RUN2 = build_gold()
check("Gold re-run (overwrite) gives identical counts", GOLD_COUNTS_RUN1 == GOLD_COUNTS_RUN2, GOLD_COUNTS_RUN2)
check("Gold tables are Delta",
      all(spark.sql(f"DESCRIBE DETAIL {tbl(GOLD_SCHEMA, t)}").first()["format"] == "delta" for t in GOLD_TABLES))

res = spark.createDataFrame(results, "check STRING, passed BOOLEAN, detail STRING")
res.show(100, False)
assert all(r[1] for r in results), "Some validation checks failed - see the table above."
print("ALL VALIDATION CHECKS PASSED")

+----------------------------------------------------------------------------------+------+---------------------------------------------------------------------------------------------------------+
|check                                                                             |passed|detail                                                                                                   |
+----------------------------------------------------------------------------------+------+---------------------------------------------------------------------------------------------------------+
|trip: csv = bronze files = bronze table = fact_trip                               |true  |4584921                                                                                                  |
|payment: csv = bronze files = bronze table = fact_payment                         |true  |1946607                                                                                                  |
|rider: cs

*Reading the result:* an unmatched station key would not be an error in itself (stations can be retired or missing from the station list), which is why the check reports how many are NULL. `duration_quality` shows how many trips have zero or negative duration; they stay in the fact table but are excluded from the duration analytics below.

## 8. Analytics
Duration queries use only `duration_quality = 'OK'` trips. `is_member` is the rider's **current** flag in the source.

In [0]:
FT, FP, DR, DS, DD = (tbl(GOLD_SCHEMA, n) for n in ["fact_trip", "fact_payment", "dim_rider", "dim_station", "dim_date"])

def age_band(col):
    return (f"CASE WHEN {col} IS NULL THEN 'unknown' WHEN {col} < 18 THEN '<18' WHEN {col} < 25 THEN '18-24' "
            f"WHEN {col} < 35 THEN '25-34' WHEN {col} < 45 THEN '35-44' WHEN {col} < 55 THEN '45-54' "
            f"WHEN {col} < 65 THEN '55-64' ELSE '65+' END")

### 8.1 Ride time by weekday and time of day

In [0]:
display(spark.sql(f"""
SELECT d.day_of_week_num, d.day_of_week_name AS weekday, f.time_band,
       COUNT(*) AS trips, ROUND(SUM(f.ride_duration_minutes), 1) AS total_minutes,
       ROUND(AVG(f.ride_duration_minutes), 2) AS avg_minutes
FROM {FT} f JOIN {DD} d ON f.start_date_key = d.date_key
WHERE f.duration_quality = 'OK'
GROUP BY d.day_of_week_num, d.day_of_week_name, f.time_band
ORDER BY d.day_of_week_num, f.time_band"""))

day_of_week_num,weekday,time_band,trips,total_minutes,avg_minutes
1,Monday,Afternoon (12-17),253691,5558861.8,21.91
1,Monday,Evening (18-23),163823,3331761.4,20.34
1,Monday,Morning (06-11),138901,2408849.1,17.34
1,Monday,Night (00-05),19673,581501.1,29.56
2,Tuesday,Afternoon (12-17),255391,4848175.0,18.98
2,Tuesday,Evening (18-23),181945,3495524.2,19.21
2,Tuesday,Morning (06-11),152370,2312466.8,15.18
2,Tuesday,Night (00-05),15054,336719.7,22.37
3,Wednesday,Afternoon (12-17),251994,4794706.7,19.03
3,Wednesday,Evening (18-23),194353,3656898.4,18.82


### 8.2 Ride time by start and end station (top 15 by average minutes, stations with ≥ 200 trips)

In [0]:
for role, key in [("start", "start_station_key"), ("end", "end_station_key")]:
    print(f"--- by {role} station ---")
    display(spark.sql(f"""
    SELECT s.station_name AS {role}_station, COUNT(*) AS trips, ROUND(AVG(f.ride_duration_minutes), 2) AS avg_minutes,
           ROUND(SUM(f.ride_duration_minutes), 0) AS total_minutes
    FROM {FT} f JOIN {DS} s ON f.{key} = s.station_key
    WHERE f.duration_quality = 'OK'
    GROUP BY s.station_name HAVING COUNT(*) >= 200
    ORDER BY avg_minutes DESC LIMIT 15"""))

--- by start station ---


start_station,trips,avg_minutes,total_minutes
Prospect Sq & 91st St,217,212.22,46053.0
Cottage Grove Ave & 83rd St,201,159.31,32021.0
Central Ave & Lake St,267,146.58,39137.0
Calumet Park,379,145.97,55322.0
State St & 95th St,382,136.57,52169.0
Sacramento Blvd & Franklin Blvd,362,94.55,34227.0
Phillips Ave & 79th St,277,93.5,25900.0
Lawndale Ave & 111th St,221,93.08,20571.0
Austin Blvd & Chicago Ave,258,78.35,20215.0
Austin Blvd & Madison St,278,77.19,21459.0


--- by end station ---


end_station,trips,avg_minutes,total_minutes
Base - 2132 W Hubbard Warehouse,468,3750.46,1755216.0
Cottage Grove Ave & 83rd St,201,153.27,30808.0
Kedzie Ave & Roosevelt Rd,359,144.8,51981.0
MLK Jr Dr & 63rd St,388,137.53,53363.0
Lincolnwood Dr & Central St,797,127.11,101307.0
Lawndale Ave & 111th St,220,116.0,25521.0
MLK Jr Dr & 56th St,306,109.33,33456.0
Prospect Sq & 91st St,240,100.81,24194.0
Central Park Ave & Ogden Ave,470,93.2,43803.0
Sacramento Blvd & Franklin Blvd,366,88.32,32324.0


### 8.3 Ride time by rider age at trip and member vs casual

In [0]:
display(spark.sql(f"""
SELECT {age_band('f.rider_age_at_trip')} AS age_band_at_trip,
       CASE WHEN r.is_member THEN 'member' ELSE 'casual' END AS rider_type,
       COUNT(*) AS trips, ROUND(AVG(f.ride_duration_minutes), 2) AS avg_minutes, ROUND(SUM(f.ride_duration_minutes), 0) AS total_minutes
FROM {FT} f JOIN {DR} r ON f.rider_key = r.rider_key
WHERE f.duration_quality = 'OK'
GROUP BY 1, 2 ORDER BY 1, 2"""))

age_band_at_trip,rider_type,trips,avg_minutes,total_minutes
18-24,casual,207362,21.33,4423182.0
18-24,member,801133,21.98,1.7612861E7
25-34,casual,299030,21.16,6328006.0
25-34,member,1216984,22.24,2.706004E7
35-44,casual,215633,20.96,4520510.0
35-44,member,848192,21.86,1.854408E7
45-54,casual,95638,22.19,2122277.0
45-54,member,378882,20.5,7768900.0
55-64,casual,32308,21.75,702641.0
55-64,member,124433,21.84,2718169.0


### 8.4 Money spent per year, quarter and month

In [0]:
display(spark.sql(f"""
SELECT d.year, d.quarter, d.month, d.month_name, COUNT(*) AS payments, ROUND(SUM(p.amount), 2) AS total_paid
FROM {FP} p JOIN {DD} d ON p.payment_date_key = d.date_key
GROUP BY ROLLUP (d.year, d.quarter, (d.month, d.month_name))
ORDER BY d.year, d.quarter, d.month""").limit(200))

year,quarter,month,month_name,payments,total_paid
null,null,null,null,1946607,19457105.25
2013,null,null,null,5351,53693.34
2013,1,null,null,82,830.65
2013,1,2,February,1,12.90
2013,1,3,March,81,817.75
2013,2,null,null,805,8164.66
2013,2,4,April,169,1672.65
2013,2,5,May,264,2716.71
2013,2,6,June,372,3775.30
2013,3,null,null,1722,17267.36


### 8.5 Money spent per member by age at account start

In [0]:
display(spark.sql(f"""
WITH per_rider AS (SELECT rider_key, SUM(amount) AS total_paid, COUNT(*) AS payments FROM {FP} GROUP BY rider_key)
SELECT {age_band('r.age_at_account_start')} AS age_band_at_account_start,
       COUNT(*) AS riders_with_payments, ROUND(SUM(p.total_paid), 2) AS total_paid,
       ROUND(AVG(p.total_paid), 2) AS avg_paid_per_rider
FROM per_rider p JOIN {DR} r ON p.rider_key = r.rider_key
GROUP BY 1 ORDER BY 1"""))

age_band_at_account_start,riders_with_payments,total_paid,avg_paid_per_rider
18-24,16808,4538310.64,270.01
25-34,23411,5978151.64,255.36
35-44,14903,3501998.69,234.99
45-54,6502,1439765.44,221.43
55-64,1778,369684.40,207.92
65+,347,61844.48,178.23
<18,9418,3567349.96,378.78


### 8.6 Extra credit: spend vs. rides per month and minutes per month
Payments and trips are aggregated **separately** to rider × month, then combined, so payment amounts are never repeated per trip. Trips only exist for a 12-month window, so payments are limited to that same window; riders with no trips or no payments in a month are kept (FULL OUTER JOIN).
Rider averages = total over the window ÷ number of months in the window.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW rider_month AS
WITH win AS (SELECT MIN(start_date_key) AS lo, MAX(start_date_key) AS hi FROM {FT}),
trip_m AS (
  SELECT f.rider_key, d.year_month, COUNT(*) AS rides, SUM(f.ride_duration_minutes) AS ride_minutes
  FROM {FT} f JOIN {DD} d ON f.start_date_key = d.date_key
  WHERE f.duration_quality = 'OK' GROUP BY f.rider_key, d.year_month),
pay_m AS (
  SELECT p.rider_key, d.year_month, SUM(p.amount) AS spend
  FROM {FP} p JOIN {DD} d ON p.payment_date_key = d.date_key CROSS JOIN win
  WHERE p.payment_date_key BETWEEN win.lo AND win.hi GROUP BY p.rider_key, d.year_month)
SELECT COALESCE(t.rider_key, p.rider_key) AS rider_key, COALESCE(t.year_month, p.year_month) AS year_month,
       COALESCE(t.rides, 0) AS rides, COALESCE(t.ride_minutes, 0.0) AS ride_minutes, COALESCE(p.spend, 0) AS spend
FROM trip_m t FULL OUTER JOIN pay_m p ON t.rider_key = p.rider_key AND t.year_month = p.year_month""")

n_months = spark.sql(f"SELECT COUNT(DISTINCT d.year_month) AS n FROM {FT} f JOIN {DD} d ON f.start_date_key = d.date_key").first()["n"]
print("months in trip window:", n_months)

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW rider_summary AS
SELECT r.rider_key, r.is_member, COALESCE(SUM(m.rides), 0) / {n_months} AS avg_rides_per_month,
       COALESCE(SUM(m.ride_minutes), 0) / {n_months} AS avg_minutes_per_month, COALESCE(SUM(m.spend), 0) AS spend_in_window
FROM {DR} r LEFT JOIN rider_month m ON r.rider_key = m.rider_key GROUP BY r.rider_key, r.is_member""")

print("--- spend by average rides per month ---")
display(spark.sql("""
SELECT CASE WHEN avg_rides_per_month = 0 THEN '0' WHEN avg_rides_per_month < 1 THEN '<1' WHEN avg_rides_per_month < 3 THEN '1-3'
            WHEN avg_rides_per_month < 6 THEN '3-6' ELSE '6+' END AS rides_per_month_band,
       COUNT(*) AS riders, ROUND(AVG(spend_in_window), 2) AS avg_spend, ROUND(SUM(spend_in_window), 2) AS total_spend
FROM rider_summary GROUP BY 1 ORDER BY 1"""))
print("--- spend by average riding minutes per month ---")
display(spark.sql("""
SELECT CASE WHEN avg_minutes_per_month = 0 THEN '0' WHEN avg_minutes_per_month < 10 THEN '<10' WHEN avg_minutes_per_month < 30 THEN '10-30'
            WHEN avg_minutes_per_month < 60 THEN '30-60' ELSE '60+' END AS minutes_per_month_band,
       COUNT(*) AS riders, ROUND(AVG(spend_in_window), 2) AS avg_spend, ROUND(SUM(spend_in_window), 2) AS total_spend
FROM rider_summary GROUP BY 1 ORDER BY 1"""))

months in trip window: 12
--- spend by average rides per month ---


rides_per_month_band,riders,avg_spend,total_spend
0,37291,83.15,3100871.42
1-3,4852,84.17,408370.16
3-6,5048,83.01,419032.90
6+,18075,83.01,1500376.26
<1,9734,83.01,808055.63


--- spend by average riding minutes per month ---


minutes_per_month_band,riders,avg_spend,total_spend
0,37291,83.15,3100871.42
10-30,4485,82.85,371593.15
30-60,3756,84.42,317086.12
60+,23043,83.20,1917267.06
<10,6425,82.47,529888.62


*How to read it:* compare `avg_spend` across the bands. Spend is the rider's payments inside the trip window; the source does not link payments to individual trips, so this is a rider-level comparison, not a per-ride cost.

## 9. Summary and re-run guidance

In [0]:
print("Bronze files :", BRONZE_DIR, "->", list(BRONZE_COLS))
print("Bronze tables:", [tbl(BRONZE_SCHEMA, n) for n in BRONZE_COLS])
print("Gold tables  :", [tbl(GOLD_SCHEMA, n) for n in GOLD_TABLES])
display(spark.createDataFrame([(n, c) for n, c in GOLD_COUNTS_RUN2.items()], "gold_table STRING, rows LONG"))

Bronze files : /Volumes/databricks_306026/default/divvy/bronze_delta -> ['rider', 'payment', 'station', 'trip']
Bronze tables: ['databricks_306026.divvy_bronze.rider', 'databricks_306026.divvy_bronze.payment', 'databricks_306026.divvy_bronze.station', 'databricks_306026.divvy_bronze.trip']
Gold tables  : ['databricks_306026.divvy_gold.dim_rider', 'databricks_306026.divvy_gold.dim_station', 'databricks_306026.divvy_gold.dim_date', 'databricks_306026.divvy_gold.fact_trip', 'databricks_306026.divvy_gold.fact_payment']


gold_table,rows
dim_rider,75000
dim_station,838
dim_date,3288
fact_trip,4584921
fact_payment,1946607


**Caveats**
* `is_member` is the rider's *current* flag; the data has no membership history, so member-vs-casual results describe today's status.
* Trips cover one year (Feb 2021 – Jan 2022); payments span 2013–2022. Time-based trip analytics and the extra-credit comparison use that overlap only.
* The source links payments to riders, not to trips.
* Zero/negative-duration trips are flagged in `duration_quality` and excluded from duration averages.
* Age uses completed years at the event date. Station IDs are strings.

**Re-running:** every write is `overwrite` (Bronze files, Bronze tables via `INSERT OVERWRITE`, Gold via `saveAsTable`), so running the notebook again replaces the data instead of adding to it; section 7 proves it by running the Gold build twice and comparing counts. Only the project's own tables/folders are touched.